In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.metrics import accuracy_score,precision_score,f1_score, classification_report, recall_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA # for dimensional reduction
import pickle # model saving
import joblib # for model saving and loading

In [2]:
df = pd.read_csv("returns_sustainability_dataset.csv")
df.head()

,Order_ID,Product_ID,User_ID,Order_Date,Product_Category,Product_Price,Order_Quantity,Discount_Applied,Shipping_Method,Payment_Method,...,Return_Status,Return_Reason,Days_to_Return,Order_Value,Return_Cost,Profit_Loss,CO2_Emissions,Packaging_Waste,CO2_Saved,Waste_Avoided
0,ORD00000,PROD0169,USER0195,2022-01-14,Clothing,1720.71,2,30.46,Next-Day,Wallet,...,Not Returned,No Return,0,2393.163468,0,2393.163468,2.0,0.4,2.0,0.4
1,ORD00001,PROD0318,USER1469,2022-01-03,Toys,744.06,5,29.62,Next-Day,Wallet,...,Returned,Size Issue,12,2618.347140,200,2418.347140,2.0,1.0,0.0,0.0
2,ORD00002,PROD0427,USER1812,2025-03-16,Clothing,983.68,5,47.80,Express,Wallet,...,Not Returned,No Return,0,2567.404800,0,2567.404800,1.5,1.0,1.5,1.0
3,ORD00003,PROD0323,USER1274,2024-11-06,Books,1855.65,2,2.90,Express,COD,...,Not Returned,No Return,0,3603.672300,0,3603.672300,1.5,0.4,1.5,0.4
4,ORD00004,PROD0325,USER0551,2023-06-07,Home Appliances,1770.97,5,44.42,Express,COD,...,Returned,Size Issue,11,4921.525630,200,4721.525630,1.5,1.0,0.0,0.0


In [3]:
df.drop(columns=["Days_to_Return","Order_ID","User_ID","Product_ID","Order_Date","CO2_Emissions","Packaging_Waste","CO2_Saved","Waste_Avoided","User_Location","Return_Reason"],inplace=True)

In [4]:
df.isnull().sum()

Product_Category    0
Product_Price       0
Order_Quantity      0
Discount_Applied    0
Shipping_Method     0
Payment_Method      0
User_Age            0
User_Gender         0
Return_Status       0
Order_Value         0
Return_Cost         0
Profit_Loss         0
dtype: int64

In [5]:
df.head()

,Product_Category,Product_Price,Order_Quantity,Discount_Applied,Shipping_Method,Payment_Method,User_Age,User_Gender,Return_Status,Order_Value,Return_Cost,Profit_Loss
0,Clothing,1720.71,2,30.46,Next-Day,Wallet,57,Female,Not Returned,2393.163468,0,2393.163468
1,Toys,744.06,5,29.62,Next-Day,Wallet,55,Female,Returned,2618.347140,200,2418.347140
2,Clothing,983.68,5,47.80,Express,Wallet,37,Female,Not Returned,2567.404800,0,2567.404800
3,Books,1855.65,2,2.90,Express,COD,47,Female,Not Returned,3603.672300,0,3603.672300
4,Home Appliances,1770.97,5,44.42,Express,COD,35,Female,Returned,4921.525630,200,4721.525630


In [6]:
#encoding:
#Product_Category Shipping_Method	Payment_Method User_Gender Return_Status
label = LabelEncoder()
df["encoded_category"] = label.fit_transform(df["Product_Category"]) 
var = dict(zip(df["Product_Category"],df["encoded_category"]))
print(var)

df["encoded_shipping"] = label.fit_transform(df["Shipping_Method"]) 
var1 = dict(zip(df["Shipping_Method"],df["encoded_shipping"]))
print(var1)

df["encoded_pay_method"] = label.fit_transform(df["Payment_Method"]) 
var2 = dict(zip(df["Payment_Method"],df["encoded_pay_method"]))
print(var2)

df["encoded_gender"] = label.fit_transform(df["User_Gender"]) 
var3 = dict(zip(df["User_Gender"],df["encoded_gender"]))
print(var3)

df["encoded_return_stat"] = label.fit_transform(df["Return_Status"]) 
var4 = dict(zip(df["Return_Status"],df["encoded_return_stat"]))
print(var4)

{'Clothing': 1, 'Toys': 4, 'Books': 0, 'Home Appliances': 3, 'Electronics': 2}
{'Next-Day': 1, 'Express': 0, 'Standard': 2}
{'Wallet': 3, 'COD': 0, 'Credit Card': 1, 'Debit Card': 2}
{'Female': 0, 'Male': 1}
{'Not Returned': 0, 'Returned': 1}


In [7]:
df.drop(columns=["Product_Category","Shipping_Method","Payment_Method","User_Gender","Return_Status"],inplace=True)

In [8]:
features = ['Discount_Applied', 'Product_Price', 'Order_Quantity', 'User_Age',
            'encoded_shipping', 'encoded_category', 'encoded_pay_method', 'encoded_gender']
x = df[features]
y = df['encoded_return_stat']

In [9]:
#data splitting:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [10]:
#scaling:
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [11]:
#pca:
pca = PCA(n_components=5)

x_train_pca = pca.fit_transform(x_train_scaled)
x_test_pca = pca.transform(x_test_scaled)

In [12]:
#hyperparameter:
import numpy as np
para_grid = {"max_depth": np.arange(1,21,2),
             "n_estimators": np.arange(50,300,50)}

In [13]:
tuning = RandomForestClassifier(random_state=42)

grid_model = GridSearchCV(param_grid=para_grid,
                          estimator=tuning,
                          n_jobs = -1,
                          scoring="accuracy",
                          cv = 5,
                          verbose = 1)
grid_model.fit(x_train_pca,y_train)

Fitting 5 folds for each of 50 candidates, totalling 250 fits


,estimator,RandomForestC...ndom_state=42)
,param_grid,"{'max_depth': array([ 1, 3..., 15, 17, 19]), 'n_estimators': array([ 50, 1...50, 200, 250])}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,np.int64(50)


In [14]:
best_model=grid_model.best_estimator_

In [15]:
grid_model.best_params_

{'max_depth': np.int64(7), 'n_estimators': np.int64(50)}

In [16]:
#model testing:
y_pred = grid_model.predict(x_test_pca)
accuracy_score(y_test,y_pred)

0.7

In [17]:
#training:
y_pred_train = grid_model.predict(x_train_pca)
accuracy_score(y_train,y_pred_train)

0.71525

In [18]:
#model saving:
pickle.dump(best_model,open("e-commerce.pkl","wb"))

In [19]:
#loading of model
loading_model = pickle.load(open("e-commerce.pkl","rb"))

In [20]:
# 1. Collect inputs safely (using float for numeric values with decimals)
product_price = int(input("Enter Product Price (e.g., 1200.50): "))
order_quantity = int(input("Enter Order Quantity (e.g., 2): "))
discount = int(input("Enter Discount Applied (e.g., 15.0): "))
age = int(input("Enter User Age (e.g., 30): "))

shipping = int(input("Enter Shipping Method {'Express': 0, 'Next-Day': 1, 'Standard': 2}: "))
category = int(input("Enter Product Category {'Books': 0, 'Clothing': 1, 'Electronics': 2, 'Home Appliances': 3, 'Toys': 4}: "))
pay_method = int(input("Enter Payment Method {'COD': 0, 'Credit Card': 1, 'Debit Card': 2, 'Wallet': 3}: "))
gender = int(input("Enter Gender {'Female': 0, 'Male': 1}: "))

# 2. Reconstruct the DataFrame with exact column names and order used during training
input_df = pd.DataFrame([[
    discount, product_price, order_quantity, age, 
    shipping, category, pay_method, gender
]], columns=features)

# 3. Apply the fitted Scaler and PCA
input_scaled = scaler.transform(input_df)
input_pca = pca.transform(input_scaled)

# 4. Predict return status using the loaded model
prediction = loading_model.predict(input_pca)[0]

print(prediction)

Enter Product Price (e.g., 1200.50):  1200
Enter Order Quantity (e.g., 2):  3
Enter Discount Applied (e.g., 15.0):  20
Enter User Age (e.g., 30):  25
Enter Shipping Method {'Express': 0, 'Next-Day': 1, 'Standard': 2}:  2
Enter Product Category {'Books': 0, 'Clothing': 1, 'Electronics': 2, 'Home Appliances': 3, 'Toys': 4}:  2
Enter Payment Method {'COD': 0, 'Credit Card': 1, 'Debit Card': 2, 'Wallet': 3}:  2
Enter Gender {'Female': 0, 'Male': 1}:  0


0
